In [ ]:
%pip install kaggle
import os
os.environ["KAGGLE_API_TOKEN"]="KGAT_e835205e3098053a244fdb243c7a2ca2"

In [ ]:
!kaggle datasets list -s lithology

ref                                                     title                                                 size  lastUpdated                 downloadCount  voteCount  usabilityRating  
------------------------------------------------------  ----------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
mpwolke/cusersmarildownloadslithocsv                    Snake River Plain - Lithology                        20773  2023-03-31 14:21:36.547000             93         12                1  
kawatgiomkar/lithology                                  Lithology                                         90996957  2024-04-17 17:20:29.410000            125          2       0.29411766  
faresazzam/well-logs-dataset-for-machine-learning       Well logs dataset for machine learning            21146630  2024-10-05 18:32:45.067000           1203         14        0.9411765  
breejeshdhar/automated-lithology-classification         Auto

In [ ]:
!kaggle datasets download -d breejeshdhar/automated-lithology-classification

Dataset URL: https://www.kaggle.com/datasets/breejeshdhar/automated-lithology-classification
License(s): CC-BY-SA-4.0
Resuming from 737148928 bytes (464350717 bytes left)...
100% 1.12G/1.12G [00:27<00:00, 17.1MB/s]



In [ ]:
!ls -lh

total 1.2G
-rw-r--r-- 1 root root 1.2G Sep  7  2023 automated-lithology-classification.zip
drwxr-xr-x 1 root root 4.0K Jun  4 13:39 sample_data


In [ ]:
!unzip -q automated-lithology-classification.zip -d lithology_dataset

In [ ]:
!find lithology_dataset -maxdepth 2 -type d

lithology_dataset
lithology_dataset/Lithology Dataset
lithology_dataset/Lithology Dataset/test
lithology_dataset/Lithology Dataset/val
lithology_dataset/Lithology Dataset/train


In [ ]:
import os
import pandas as pd

dataset_path = "/content/lithology_dataset/Lithology Dataset/train"

data = []

for rock_class in os.listdir(dataset_path):
    class_path = os.path.join(dataset_path, rock_class)

    if os.path.isdir(class_path):
        for file in os.listdir(class_path):
            if file.lower().endswith((".jpg", ".jpeg", ".png")):
                data.append({
                    "image": os.path.join(class_path, file),
                    "label": rock_class
                })

df = pd.DataFrame(data)
df.head()

,image,label
0,/content/lithology_dataset/Lithology Dataset/t...,shale
1,/content/lithology_dataset/Lithology Dataset/t...,shale
2,/content/lithology_dataset/Lithology Dataset/t...,shale
3,/content/lithology_dataset/Lithology Dataset/t...,shale
4,/content/lithology_dataset/Lithology Dataset/t...,shale


In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))

Train: 57596
Validation: 14399


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    rescale=1./255
)

In [ ]:
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col="image",
    y_col="label",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical"
)

val_generator = val_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col="image",
    y_col="label",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical"
)

Found 57596 validated image filenames belonging to 4 classes.
Found 14399 validated image filenames belonging to 4 classes.


In [ ]:
import os
print(os.listdir('/content/lithology_dataset/Lithology Dataset/train'))

['shale', 'limestone', 'garbage', 'sandstone']


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense, Dropout

model = Sequential()

# First Convolution Block
model.add(Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)))
model.add(MaxPooling2D(pool_size=(2,2)))

# Second Convolution Block
model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))

# Third Convolution Block
model.add(Conv2D(128, (3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))

# Flatten
model.add(Flatten())

# Fully Connected Layer
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

# Output Layer
model.add(Dense(4, activation='softmax'))

# Display model summary
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,476 (42.61 MB)

 Trainable params: 11,169,476 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5
)

Epoch 1/5
1800/1800 ━━━━━━━━━━━━━━━━━━━━ 739s 406ms/step - accuracy: 0.8393 - loss: 0.4357 - val_accuracy: 0.9190 - val_loss: 0.2183
Epoch 2/5
1800/1800 ━━━━━━━━━━━━━━━━━━━━ 709s 394ms/step - accuracy: 0.9123 - loss: 0.2630 - val_accuracy: 0.9256 - val_loss: 0.2060
Epoch 3/5
1800/1800 ━━━━━━━━━━━━━━━━━━━━ 702s 390ms/step - accuracy: 0.9275 - loss: 0.2166 - val_accuracy: 0.9290 - val_loss: 0.1913
Epoch 4/5
1800/1800 ━━━━━━━━━━━━━━━━━━━━ 701s 390ms/step - accuracy: 0.9377 - loss: 0.1922 - val_accuracy: 0.9553 - val_loss: 0.1278
Epoch 5/5
1800/1800 ━━━━━━━━━━━━━━━━━━━━ 698s 388ms/step - accuracy: 0.9442 - loss: 0.1711 - val_accuracy: 0.9464 - val_loss: 0.1482


In [3]:
from google.colab import drive
drive.mount('/content/drive')

model.save('/content/drive/MyDrive/rock_classification.h5')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


NameError: name 'model' is not defined